# Gated residual IQ + spectrogram modulation classifier

This experiment keeps the earlier single-representation notebooks intact. It trains a fused model using the repository's two EfficientNet-B0 implementations:

```text
IQ [2, N] ──────────> EfficientNet1D ──> IQ logits ───────────────┐
                                                                  ├─> class logits
log spectrogram [1,F,T] -> EfficientNet2D -> gated correction ───┘
```

Both views come from the **same stored IQ example**. Each branch first learns its own classification task. Fusion then starts as the exact pretrained IQ classifier and learns a gated, zero-initialized correction from spectrogram logits. The IQ branch remains frozen; only the fusion modules and, in the final stage, the last spectrogram feature blocks are trained. The notebook reuses data created by `train_iq_vs_spectrogram_modrec.ipynb` when available and can generate the same splits otherwise. A CUDA GPU is assumed.

In [ ]:
from __future__ import annotations

import copy
import random
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from torchsig_models.models.iq_models.efficientnet import efficientnet_b0 as iq_efficientnet_b0
from torchsig_models.models.spectrogram_models.efficientnet import efficientnet_b0 as spectrogram_efficientnet_b0
from torchsig.datasets.datasets import StaticTorchSigDataset, TorchSigIterableDataset
from torchsig.transforms.transforms import ComplexTo2D
from torchsig.utils.data_loading import WorkerSeedingDataLoader
from torchsig.utils.defaults import TorchSigDefaults
from torchsig.utils.writer import DatasetCreator

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
if not torch.cuda.is_available():
    raise RuntimeError("This notebook is configured for a CUDA GPU.")
device = torch.device("cuda")
print(torch.cuda.get_device_name(), torch.__version__)

In [ ]:
SIGNALS = ["bpsk", "qpsk", "16qam", "2fsk", "4fsk", "lfm-radar"]
NUM_CLASSES = len(SIGNALS)
SPLIT_SIZES = {"train": 12_000, "val": 2_000, "test": 4_000}
SPLIT_SEEDS = {"train": 3101, "val": 3102, "test": 3103}
DATA_ROOT = Path("runs/iq_vs_spectrogram_modrec/data")
CHECKPOINT_PATH = Path("runs/iq_vs_spectrogram_modrec/gated_residual_fused_efficientnet_b0.pt")
REGENERATE = False
NUM_IQ_SAMPLES = 4096
FFT_SIZE, FFT_HOP = 128, 32
BATCH_SIZE, NUM_WORKERS = 64, 4
PRETRAIN_EPOCHS, HEAD_EPOCHS, FINETUNE_EPOCHS = 15, 5, 10
PRETRAIN_LR, HEAD_LR, FINETUNE_LR = 3e-3, 2e-3, 2e-4
WEIGHT_DECAY = 1e-4
UNFROZEN_BLOCKS = 2
SNR_BINS = [-10, 0, 5, 10, 15, 20, 30]

metadata = TorchSigDefaults().default_dataset_metadata.copy()
metadata.update({
    "sample_rate": 10_000_000, "num_iq_samples_dataset": NUM_IQ_SAMPLES,
    "num_signals_min": 1, "num_signals_max": 1,
    "cochannel_overlap_probability": 0.0, "snr_db_min": -5.0,
    "snr_db_max": 25.0, "noise_power_db": 0.0,
    "signal_duration_in_samples_min": 3600,
    "signal_duration_in_samples_max": NUM_IQ_SAMPLES,
    "bandwidth_min": 750_000, "bandwidth_max": 2_500_000,
    "signal_center_freq_min": -2_500_000, "signal_center_freq_max": 2_500_000,
    "frequency_min": -5_000_000, "frequency_max": 5_000_000,
})

## Reuse or generate the shared IQ splits

Generation settings and seeds match the two-model notebook. `REGENERATE=False` protects existing static data.

In [ ]:
def create_split(name, length, seed):
    root = DATA_ROOT / name
    if root.exists() and not REGENERATE:
        print(f"Reusing {root}")
        return root
    source = TorchSigIterableDataset(
        metadata=metadata.copy(), seed=seed, transforms=[ComplexTo2D()],
        signal_generators=SIGNALS, target_labels=["class_index", "snr_db"],
    )
    generation_loader = WorkerSeedingDataLoader(
        source, batch_size=64, num_workers=NUM_WORKERS, collate_fn=list
    )
    DatasetCreator(
        dataloader=generation_loader, root=str(root), overwrite=REGENERATE, dataset_length=length
    ).create()
    return root

split_roots = {name: create_split(name, size, SPLIT_SEEDS[name]) for name, size in SPLIT_SIZES.items()}
raw_splits = {
    name: StaticTorchSigDataset(root=str(root), target_labels=["class_index", "snr_db"])
    for name, root in split_roots.items()
}
print({name: len(dataset) for name, dataset in raw_splits.items()})

## Load IQ and create spectrograms in GPU batches

A shared random phase rotation augments IQ during training. Spectrograms are derived later with a batched GPU STFT, avoiding thousands of small CPU STFT calls in DataLoader workers. This preserves exact pairing and makes the first epoch start much faster.

In [ ]:
def unpack_target(target):
    label, snr = target
    return int(np.asarray(label).reshape(-1)[0]), float(np.asarray(snr).reshape(-1)[0])

class FusedRepresentationDataset(Dataset):
    def __init__(self, base, augment=False):
        self.base, self.augment = base, augment
    def __len__(self):
        return len(self.base)
    def __getitem__(self, index):
        iq, target = self.base[index]
        iq = torch.as_tensor(iq, dtype=torch.float32)
        label, snr = unpack_target(target)
        if not 0 <= label < NUM_CLASSES:
            raise ValueError(f"Unexpected class index {label}")
        if self.augment:
            phase = torch.rand(()) * (2 * torch.pi)
            i, q = iq[0].clone(), iq[1].clone()
            iq = torch.stack((i * phase.cos() - q * phase.sin(), i * phase.sin() + q * phase.cos()))
        iq = iq / iq.square().mean().sqrt().clamp_min(1e-6)
        return iq, label, snr

def make_spectrogram(iq):
    """Create normalized log-power spectrograms for a GPU IQ batch."""
    z = torch.complex(iq[:, 0], iq[:, 1])
    window = torch.hann_window(FFT_SIZE, device=iq.device, dtype=iq.dtype)
    stft = torch.stft(
        z, n_fft=FFT_SIZE, hop_length=FFT_HOP, win_length=FFT_SIZE,
        window=window, center=False, return_complex=True,
    )
    spectrogram = torch.log1p(stft.abs().square()).unsqueeze(1)
    mean = spectrogram.mean(dim=(-2, -1), keepdim=True)
    std = spectrogram.std(dim=(-2, -1), keepdim=True).clamp_min(1e-6)
    return (spectrogram - mean) / std

datasets = {
    split: FusedRepresentationDataset(raw, augment=split == "train")
    for split, raw in raw_splits.items()
}
loaders = {
    split: DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=split == "train", num_workers=NUM_WORKERS,
        pin_memory=True, persistent_workers=NUM_WORKERS > 0,
    ) for split, dataset in datasets.items()
}
iq, label, snr = datasets["train"][0]
preview_spectrogram = make_spectrogram(iq.unsqueeze(0).to(device)).cpu()
print(iq.shape, preview_spectrogram.shape, label, snr)

## Build the branch and gated residual models

Each branch retains its supervised classifier. The fusion model adds a learned correction derived from spectrogram logits to the IQ logits. Both the correction projection and gate output weights are initialized to zero, making the initial fused output exactly equal to the pretrained IQ output.

In [ ]:
def make_branch(representation, num_classes):
    if representation == "iq":
        return iq_efficientnet_b0(
            num_classes=num_classes, drop_path_rate=0.1, drop_rate=0.1
        )
    if representation == "spectrogram":
        return spectrogram_efficientnet_b0(
            num_classes=num_classes, input_channels=1,
            drop_path_rate=0.1, drop_rate=0.1, normalize=False,
        )
    raise ValueError(f"Unknown representation: {representation}")

class GatedResidualFusion(nn.Module):
    def __init__(self, iq_backbone, spectrogram_backbone, num_classes):
        super().__init__()
        self.iq_backbone = iq_backbone
        self.spectrogram_backbone = spectrogram_backbone
        self.correction = nn.Linear(num_classes, num_classes)
        self.gate = nn.Sequential(
            nn.Linear(2 * num_classes, 16), nn.SiLU(), nn.Linear(16, 1), nn.Sigmoid()
        )
        nn.init.zeros_(self.correction.weight); nn.init.zeros_(self.correction.bias)
        nn.init.zeros_(self.gate[2].weight); nn.init.constant_(self.gate[2].bias, -4.0)
    def forward(self, iq, spectrogram):
        iq_logits = self.iq_backbone(iq)
        spectrogram_logits = self.spectrogram_backbone(spectrogram)
        gate = self.gate(torch.cat((iq_logits, spectrogram_logits), dim=1))
        return iq_logits + gate * self.correction(spectrogram_logits)

branches = {representation: make_branch(representation, NUM_CLASSES) for representation in ("iq", "spectrogram")}
for representation, branch in branches.items():
    print(f"{representation:12}: {sum(parameter.numel() for parameter in branch.parameters()):,} parameters")

## Staged training

First, independently pretrain both classifiers. Next, train only the zero-initialized correction and gate with frozen backbones. Finally, unfreeze the last spectrogram feature blocks and fine-tune them with a ten-times lower learning rate. The IQ branch always remains frozen and all frozen batch-normalization layers remain in evaluation mode. Checkpoint selection includes the initial IQ-equivalent state, so a learned fusion state is retained only when it improves validation accuracy.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

def branch_input(iq, representation):
    return iq if representation == "iq" else make_spectrogram(iq)

@torch.inference_mode()
def branch_accuracy(model, loader, representation):
    model.eval(); correct = total = 0
    for iq, target, _ in loader:
        iq = iq.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        prediction = model(branch_input(iq, representation)).argmax(1)
        correct += (prediction == target).sum().item(); total += target.numel()
    return correct / total

def pretrain_branch(model, representation):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, PRETRAIN_EPOCHS)
    scaler = torch.amp.GradScaler("cuda")
    best_accuracy, best_state = -1.0, None
    for epoch in range(1, PRETRAIN_EPOCHS + 1):
        model.train(); loss_sum = examples = 0
        for iq, target, _ in loaders["train"]:
            iq = iq.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                loss = criterion(model(branch_input(iq, representation)), target)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            loss_sum += loss.item() * target.numel(); examples += target.numel()
        scheduler.step()
        val_accuracy = branch_accuracy(model, loaders["val"], representation)
        if val_accuracy > best_accuracy:
            best_accuracy, best_state = val_accuracy, copy.deepcopy(model.state_dict())
        print(f"{representation} pretrain {epoch:02}/{PRETRAIN_EPOCHS}: loss={loss_sum/examples:.4f}, val_acc={val_accuracy:.3%}")
    model.load_state_dict(best_state)
    return model, best_accuracy

pretrain_started = time.perf_counter()
branch_validation = {}
for representation in ("iq", "spectrogram"):
    branches[representation], branch_validation[representation] = pretrain_branch(
        branches[representation], representation
    )
print("Best branch validation accuracy:", branch_validation)

@torch.inference_mode()
def collect_branch_predictions(model, loader, representation):
    model.eval(); predictions, targets, snrs = [], [], []
    for iq, target, snr in loader:
        iq = iq.to(device, non_blocking=True)
        logits = model(branch_input(iq, representation))
        predictions.append(logits.argmax(1).cpu())
        targets.append(target); snrs.append(torch.as_tensor(snr))
    return torch.cat(predictions).numpy(), torch.cat(targets).numpy(), torch.cat(snrs).numpy()

iq_prediction, test_target, test_snr = collect_branch_predictions(
    branches["iq"], loaders["test"], "iq"
)
print(f"Restored IQ branch test accuracy: {(iq_prediction == test_target).mean():.3%}")

model = GatedResidualFusion(branches["iq"], branches["spectrogram"], NUM_CLASSES).to(device)
with torch.inference_mode():
    print("Fused output shape:", model(iq.unsqueeze(0).to(device), preview_spectrogram.to(device)).shape)

@torch.inference_mode()
def accuracy(model, loader):
    model.eval(); correct = total = 0
    for iq, target, _ in loader:
        iq = iq.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        correct += (model(iq, make_spectrogram(iq)).argmax(1) == target).sum().item()
        total += target.numel()
    return correct / total

def freeze_for_head_training(model):
    for backbone in (model.iq_backbone, model.spectrogram_backbone):
        backbone.requires_grad_(False); backbone.eval()
    model.correction.requires_grad_(True); model.gate.requires_grad_(True)

def unfreeze_spectrogram_blocks(model, block_count):
    model.iq_backbone.requires_grad_(False); model.iq_backbone.eval()
    model.spectrogram_backbone.requires_grad_(False); model.spectrogram_backbone.eval()
    blocks = list(model.spectrogram_backbone.model.blocks.children())
    if not blocks:
        raise ValueError("Spectrogram EfficientNet backbone has no feature blocks")
    for block in blocks[-block_count:]:
        block.requires_grad_(True); block.train()
    for module in (model.spectrogram_backbone.model.conv_head, model.spectrogram_backbone.model.bn2):
        module.requires_grad_(True); module.train()
    model.correction.requires_grad_(True); model.gate.requires_grad_(True)

def set_stage_mode(model, fine_tune):
    model.train(); model.correction.train(); model.gate.train()
    model.iq_backbone.eval(); model.spectrogram_backbone.eval()
    if fine_tune:
        for block in list(model.spectrogram_backbone.model.blocks.children())[-UNFROZEN_BLOCKS:]:
            block.train()
        model.spectrogram_backbone.model.conv_head.train()
        model.spectrogram_backbone.model.bn2.train()

def train_fusion_stage(model, name, epochs, learning_rate, fine_tune=False):
    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr=learning_rate, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    scaler = torch.amp.GradScaler("cuda")
    best_accuracy, best_state = -1.0, None
    history = defaultdict(list)
    for epoch in range(1, epochs + 1):
        set_stage_mode(model, fine_tune); loss_sum = examples = 0
        for iq, target, _ in loaders["train"]:
            iq = iq.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                loss = criterion(model(iq, make_spectrogram(iq)), target)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            loss_sum += loss.item() * target.numel(); examples += target.numel()
        scheduler.step()
        val_accuracy = accuracy(model, loaders["val"])
        history["train_loss"].append(loss_sum / examples)
        history["val_accuracy"].append(val_accuracy)
        if val_accuracy > best_accuracy:
            best_accuracy, best_state = val_accuracy, copy.deepcopy(model.state_dict())
        print(f"{name} {epoch:02}/{epochs}: loss={loss_sum/examples:.4f}, val_acc={val_accuracy:.3%}")
    model.load_state_dict(best_state)
    return dict(history), best_accuracy, best_state

freeze_for_head_training(model)
baseline_accuracy = accuracy(model, loaders["val"])
baseline_state = copy.deepcopy(model.state_dict())
print(f"IQ-equivalent initial validation accuracy: {baseline_accuracy:.3%}")
head_history, head_accuracy, head_state = train_fusion_stage(
    model, "fusion head", HEAD_EPOCHS, HEAD_LR
)
unfreeze_spectrogram_blocks(model, UNFROZEN_BLOCKS)
finetune_history, finetune_accuracy, finetune_state = train_fusion_stage(
    model, "fine-tune", FINETUNE_EPOCHS, FINETUNE_LR, fine_tune=True
)
best_accuracy, best_state = max(
    ((baseline_accuracy, baseline_state), (head_accuracy, head_state),
     (finetune_accuracy, finetune_state)), key=lambda result: result[0]
)
model.load_state_dict(best_state)
history = {"head": head_history, "fine_tune": finetune_history}
training_seconds = time.perf_counter() - pretrain_started
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "model_state_dict": best_state, "signals": SIGNALS,
    "branch_validation_accuracy": branch_validation,
    "fused_validation_accuracy": best_accuracy,
}, CHECKPOINT_PATH)
print(f"Best fused validation accuracy: {best_accuracy:.3%}")
print(f"Total training time: {training_seconds/60:.1f} minutes")

## Evaluate confusion and SNR behavior

In [ ]:
@torch.inference_mode()
def collect_predictions(model, loader):
    model.eval(); predictions, targets, snrs = [], [], []
    for iq, target, snr in loader:
        iq = iq.to(device, non_blocking=True)
        logits = model(iq, make_spectrogram(iq))
        predictions.append(logits.argmax(1).cpu()); targets.append(target); snrs.append(torch.as_tensor(snr))
    return torch.cat(predictions).numpy(), torch.cat(targets).numpy(), torch.cat(snrs).numpy()

prediction, target, snr = collect_predictions(model, loaders["test"])
assert np.array_equal(target, test_target)
assert np.allclose(snr, test_snr)
iq_correct = iq_prediction == target
fused_correct = prediction == target
iq_accuracy = iq_correct.mean()
overall_accuracy = fused_correct.mean()
wrong_to_right = np.count_nonzero(~iq_correct & fused_correct)
right_to_wrong = np.count_nonzero(iq_correct & ~fused_correct)
print(f"Paired IQ test accuracy:    {iq_accuracy:.2%}")
print(f"Paired fused test accuracy: {overall_accuracy:.2%}")
print(f"Paired improvement:         {(overall_accuracy - iq_accuracy):+.2%}")
print(f"Wrong to right: {wrong_to_right}; right to wrong: {right_to_wrong}")
print("Per-class recall change (IQ -> fused):")
for class_index, class_name in enumerate(SIGNALS):
    selected = target == class_index
    iq_recall = iq_correct[selected].mean()
    fused_recall = fused_correct[selected].mean()
    print(f"  {class_name:10}: {iq_recall:.2%} -> {fused_recall:.2%} ({fused_recall - iq_recall:+.2%})")

matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
np.add.at(matrix, (target, prediction), 1)
normalized = matrix / matrix.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
image = axes[0].imshow(normalized, vmin=0, vmax=1, cmap="Blues")
axes[0].set(title=f"Gated residual accuracy={overall_accuracy:.2%}", xlabel="Predicted", ylabel="True")
axes[0].set_xticks(range(NUM_CLASSES), SIGNALS, rotation=45, ha="right")
axes[0].set_yticks(range(NUM_CLASSES), SIGNALS)
for row in range(NUM_CLASSES):
    for col in range(NUM_CLASSES):
        axes[0].text(col, row, f"{normalized[row, col]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=axes[0], shrink=.8)
bin_accuracy, bin_centers = [], []
for lower, upper in zip(SNR_BINS[:-1], SNR_BINS[1:]):
    selected = (snr >= lower) & (snr < upper)
    if selected.any():
        bin_accuracy.append((prediction[selected] == target[selected]).mean())
        bin_centers.append((lower + upper) / 2)
axes[1].plot(bin_centers, bin_accuracy, marker="o")
axes[1].set(title="Gated residual accuracy by SNR", xlabel="SNR (dB)", ylabel="Accuracy", ylim=(0, 1.02))
axes[1].grid(alpha=.25)
plt.tight_layout(); plt.show()

## What success—and failure—would mean

A useful residual should retain IQ's PSK/QAM separation while selectively using spectrogram evidence. Report both the IQ-equivalent initial validation accuracy and the selected fused validation accuracy: if they are equal, validation rejected every learned correction. If validation improves but test accuracy does not, repeat with multiple seeds before attributing the difference to fusion.

The fused network is roughly twice as expensive as either branch, so any accuracy improvement should be evaluated against latency and memory on target hardware. Repeat multiple seeds before drawing a conclusion.